# 🧩 Mini-Lab: ReAct Pattern

**Module 4: Prompt Engineering & Security** | **Duration: ~30 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** the ReAct (Reason + Act) prompting pattern and how it enables agent-like behavior
2. **Implement** a ReAct loop where the LLM reasons, selects tools, and acts on observations
3. **Recognize** how the Thought → Action → Observation cycle gives LLMs the ability to use external tools

## Target Concepts

| Concept | Description |
|---------|-------------|
| ReAct Prompting | The Reason + Act pattern where the model alternates between reasoning about what to do and taking actions (tool calls), then observing results — enabling agent-like multi-step problem solving |

## What Is ReAct?

**ReAct** stands for **Reason + Act**. It's a prompting pattern where the LLM follows a structured loop:

1. **Thought** — The model reasons about what it knows and what it needs to do next
2. **Action** — The model picks a tool/action and provides the input
3. **Observation** — The result from the tool is fed back to the model
4. **Repeat** — Until the model has enough information to give a final answer

This is the foundation of how modern AI agents work. Instead of answering in one shot, the model can **plan, use tools, and refine** its answer over multiple steps.

## Setup

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # Uses OPENAI_API_KEY from .env

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Step 1: Define Simple Tools

For the ReAct pattern to work, the model needs **tools** it can call. We'll create three simple simulated tools that a model might use to answer a question it can't answer from memory alone.

In [3]:
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def lookup(topic: str) -> str:
    """Simulated knowledge lookup (like a search engine)."""
    data = {
        "population of france": "The population of France is approximately 68 million (2024).",
        "population of germany": "The population of Germany is approximately 84 million (2024).",
        "area of france": "France covers approximately 551,695 square kilometers.",
        "area of germany": "Germany covers approximately 357,022 square kilometers.",
    }
    result = data.get(topic.lower().strip(), f"No data found for '{topic}'.")
    return result

# Map tool names to functions
TOOLS = {
    "calculator": calculator,
    "lookup": lookup,
}

print("Tools defined:", list(TOOLS.keys()))

Tools defined: ['calculator', 'lookup']


## Step 2: The ReAct System Prompt

The key to ReAct is a **carefully structured system prompt** that teaches the model the Thought → Action → Observation loop and tells it what tools are available.

In [4]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step using tools.

You have access to the following tools:
- lookup[topic]: Looks up factual information about a topic. Input should be a short phrase like "population of france".
- calculator[expression]: Evaluates a math expression. Input should be a valid Python math expression like "68000000 / 551695".

To solve a problem, you MUST follow this exact format for EACH step:

Thought: <your reasoning about what to do next>
Action: <tool_name>[<input>]

After you receive an Observation (the tool result), continue with another Thought/Action if needed.

When you have enough information, respond with:
Thought: I now have enough information to answer.
Final Answer: <your complete answer>

IMPORTANT RULES:
- Always start with a Thought before any Action.
- Use EXACTLY one Action per step.
- Wait for the Observation before your next Thought.
- Do NOT make up information — use lookup for facts and calculator for math.
"""

md("**ReAct system prompt defined.** The model now knows about the Thought → Action → Observation loop.")

**ReAct system prompt defined.** The model now knows about the Thought → Action → Observation loop.

## Step 3: Build the ReAct Loop

Now we build the actual loop. The logic is:

1. Send the question to the model with the ReAct system prompt
2. Parse the model's response for an **Action** (tool call)
3. Execute the tool and append the **Observation** back into the conversation
4. Repeat until the model outputs a **Final Answer**

In [5]:
import re

def parse_action(text: str):
    """Extract tool name and input from an Action line like: Action: lookup[population of france]"""
    match = re.search(r"Action:\s*(\w+)\[(.+?)\]", text)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return None, None

def react_loop(question: str, max_steps: int = 5):
    """Run the ReAct loop for a given question."""
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    
    print(f"❓ Question: {question}\n")
    print("=" * 60)
    
    for step in range(1, max_steps + 1):
        # Ask the model
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
            max_tokens=300,
        )
        assistant_text = response.choices[0].message.content.strip()
        messages.append({"role": "assistant", "content": assistant_text})
        
        print(f"\n🔄 Step {step}:")
        print(assistant_text)
        
        # Check if we have a Final Answer
        if "Final Answer:" in assistant_text:
            print("\n" + "=" * 60)
            final = assistant_text.split("Final Answer:")[-1].strip()
            print(f"✅ Final Answer: {final}")
            return final
        
        # Parse the action
        tool_name, tool_input = parse_action(assistant_text)
        
        if tool_name and tool_name in TOOLS:
            # Execute the tool
            observation = TOOLS[tool_name](tool_input)
            print(f"\n📌 Observation: {observation}")
            
            # Feed observation back to the model
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        elif tool_name:
            observation = f"Error: Unknown tool '{tool_name}'. Available: {list(TOOLS.keys())}"
            print(f"\n⚠️ {observation}")
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            # No action found — nudge the model
            messages.append({"role": "user", "content": "Please continue with a Thought and Action, or provide a Final Answer."})
    
    print("\n⏱️ Max steps reached without a final answer.")
    return None

## Step 4: Test the ReAct Pattern

Let's ask a question that requires **multiple tool calls** — the model needs to look up facts and then do math. This is something the model can't reliably do in a single step.

In [6]:
answer = react_loop(
    "Which country has a higher population density — France or Germany? "
    "Show the density for each."
)

❓ Question: Which country has a higher population density — France or Germany? Show the density for each.


🔄 Step 1:
Thought: To compare the population density of France and Germany, I need to find the population and area of each country. Then, I can calculate the population density using the formula: population density = population / area. I'll start by looking up the population and area of France. 
Action: lookup[population and area of France]

📌 Observation: No data found for 'population and area of France'.

🔄 Step 2:
Thought: It seems that the lookup did not return the necessary information. I will first look up the population of France. 
Action: lookup[population of France]

📌 Observation: The population of France is approximately 68 million (2024).

🔄 Step 3:
Thought: Now that I have the population of France, I need to find the area of France to calculate its population density. I'll look up the area of France next. 
Action: lookup[area of France]

📌 Observation: France covers 

Notice the pattern in the output above:
- The model **thinks** about what information it needs
- It **acts** by calling a tool (lookup or calculator)
- It **observes** the result and decides what to do next
- It repeats until it can give a **Final Answer**

This is the core ReAct loop — and it's exactly how modern AI agents (like those built with LangChain or OpenAI function calling) work under the hood.

## Step 5: A Simpler Question (Fewer Steps)

Let's see how the model handles a question that requires fewer tool calls.

In [7]:
answer = react_loop("What is the population of Germany?")

❓ Question: What is the population of Germany?


🔄 Step 1:
Thought: I need to look up the current population of Germany to provide an accurate answer. 
Action: lookup[population of Germany]

📌 Observation: The population of Germany is approximately 84 million (2024).

🔄 Step 2:
Thought: I now have enough information to answer.
Final Answer: The population of Germany is approximately 84 million as of 2024.

✅ Final Answer: The population of Germany is approximately 84 million as of 2024.


With a simple factual question, the model only needed one lookup and one step. The ReAct pattern naturally adapts to the complexity of the task.

## Step 6: Inspect the Conversation Trace

Let's run one more example and look at the full message history to see exactly how the conversation is structured between the model and the tools.

In [8]:
def react_loop_with_trace(question: str, max_steps: int = 5):
    """Run ReAct loop and return the full message trace."""
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
            max_tokens=300,
        )
        assistant_text = response.choices[0].message.content.strip()
        messages.append({"role": "assistant", "content": assistant_text})
        
        if "Final Answer:" in assistant_text:
            return messages
        
        tool_name, tool_input = parse_action(assistant_text)
        if tool_name and tool_name in TOOLS:
            observation = TOOLS[tool_name](tool_input)
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            messages.append({"role": "user", "content": "Please continue with a Thought and Action, or provide a Final Answer."})
    
    return messages

trace = react_loop_with_trace("What is 68000000 divided by 551695? Round to the nearest whole number.")

# Display the trace (skip system prompt for readability)
output = "### Conversation Trace\n\n"
for msg in trace[1:]:  # skip system prompt
    role = msg["role"].upper()
    output += f"**{role}:**\n\n{msg['content']}\n\n---\n\n"

md(output)

### Conversation Trace

**USER:**

What is 68000000 divided by 551695? Round to the nearest whole number.

---

**ASSISTANT:**

Thought: I need to calculate the division of 68000000 by 551695 and then round the result to the nearest whole number. 
Action: calculator[68000000 / 551695]

---

**USER:**

Observation: 123.25650948440715

---

**ASSISTANT:**

Thought: Now that I have the result of the division, I need to round 123.25650948440715 to the nearest whole number. 
Action: calculator[round(123.25650948440715)]

---

**USER:**

Observation: Error: name 'round' is not defined

---

**ASSISTANT:**

Thought: It seems I cannot use the round function directly in the calculator. Instead, I will manually round the number 123.25650948440715 to the nearest whole number, which is 123. 
Final Answer: 123

---



The trace shows the exact back-and-forth:
- **User** sends the question
- **Assistant** responds with Thought + Action
- **User** (our code) injects the Observation
- **Assistant** continues until it reaches a Final Answer

This is the **orchestration pattern** that powers AI agents: the LLM decides *which* tool to use, our code *executes* the tool, and the result flows back into the conversation.

## 🎯 Summary

### Key Takeaways

1. **ReAct (Reason + Act)** — a prompting pattern where the model alternates between thinking (Thought), using tools (Action), and processing results (Observation)
2. **Thought → Action → Observation loop** — the model plans each step, calls a tool, sees the result, and decides what to do next until it has a final answer
3. **Agent foundation** — ReAct is the core pattern behind modern AI agents; the LLM decides *what* to do while external code *executes* the actions
4. **Adaptive complexity** — the model naturally uses more or fewer steps depending on the difficulty of the question